# Emergency Department EDA

**R²D² workshop · 15-minute learner exercise**

This notebook follows four sections:

1. **Setup** — load the data and understand what one row represents.
2. **Question 1: Data cleaning** — identify missing values and exact duplicates.
3. **Question 2: Data manipulation** — create arrival hour and ED length of stay.
4. **Question 3: Descriptive analysis and visualisation** — summarise distributions, draw charts, and explore correlation.

Run cells from top to bottom. In learner exercise cells, replace each `____` with one Python expression. A `NameError` mentioning `____` means a gap remains.

## Exercise objective

By the end of Sections 1–4, learners should be able to:

1. distinguish stored rows, ED encounters, and unique patients;
2. identify missing values and remove only exact duplicate copies;
3. convert timestamp text and derive arrival hour and ED length of stay (LOS);
4. describe distributions using counts, mean, median, quartiles, and charts;
5. interpret a correlation without claiming that it proves causation.

**Clinical question:** When do patients arrive, how long do they remain in ED, and are the data suitable for describing patient flow?

Sections 1–4 form the 15-minute core exercise. Section 5 contains optional, ready-to-run examples for further learning.

## Section 1 — Setup

### 1.1 Download and load the data · Run this

**Google Colab:** the next cell downloads all three workbooks from the [workshop dataset repository](https://github.com/chitinglow/R2D2_DataSet) into `/content/data/`:

- Emergency Department: `ae_synthetic_data_2024_2025.xlsx`
- Inpatient: `inpatient_synthetic_data_2024_2025.xlsx`
- Outpatient: `outpatient_synthetic_data_2024_Mar2026.xlsx`

Colab storage is temporary, so rerun setup after a runtime reset.

**Local Jupyter:** open the notebook from the project folder. The ED workbook is read from `data/`.

In [1]:
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError
import sys
import importlib.util
import subprocess
import pandas as pd
import altair as alt
from IPython.display import display

# Colab connection check and package installation
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    for package in ["openpyxl", "altair"]:
        if importlib.util.find_spec(package) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Data download and setup
DATA_URL = "https://raw.githubusercontent.com/chitinglow/R2D2_DataSet/main/"
DATA_FILES = {
    "ED": "ae_synthetic_data_2024_2025.xlsx",
    "Inpatient": "inpatient_synthetic_data_2024_2025.xlsx",
    "Outpatient": "outpatient_synthetic_data_2024_Mar2026.xlsx",
}
data_dir = Path("/content/data") if IN_COLAB else Path("data")
dataset_paths = {name: data_dir / filename for name, filename in DATA_FILES.items()}

if IN_COLAB:
    data_dir.mkdir(parents=True, exist_ok=True)
    # Fetch all three workbooks each time setup runs.
    for name, filename in DATA_FILES.items():
        try:
            with urlopen(DATA_URL + filename, timeout=30) as response:
                workbook_bytes = response.read()
        except (URLError, TimeoutError) as error:
            raise ConnectionError(
                f"Could not download {name} data ({filename}). "
                "Check your internet connection and DATA_URL, then rerun this cell."
            ) from error
        dataset_paths[name].write_bytes(workbook_bytes)
        print(f"Downloaded {name}: {dataset_paths[name]}")
    print("All three datasets are ready in /content/data.")

# The ED exercise uses only the ED workbook.
# Inpatient and outpatient workbooks are saved for the other workshop sections.
# Change the path below if you want to use a different workbooks
data_source = dataset_paths["ED"]
if not data_source.exists():
    raise FileNotFoundError(f"Missing {data_source}. Open Jupyter from the project folder.")

# Load the raw data from the ED workbook
df_raw = pd.read_excel(data_source, sheet_name="AE_Data", dtype="string")
print(f"Loaded {len(df_raw):,} rows and {len(df_raw.columns)} columns.")

Loaded 3,002 rows and 30 columns.


## Reference — Download the DataCamp pandas cheat sheet

Use the official [DataCamp pandas cheat sheet](!https://www.datacamp.com/cheat-sheet/pandas-cheat-sheet-for-data-science-in-python) for additional syntax examples. The webpage is saved locally for offline reference.

For additional information refers to the official [Pandas website](https://pandas.pydata.org/pandas-docs/stable/user_guide/index.html#user-guide)


### 1.2 Understand the rows

A **row** is a stored record, an **encounter** is one ED visit, and one **patient** can have several encounters. Compare these counts before choosing a denominator.

In [8]:
# Display summary statistics for the raw data
display(pd.Series({
    "Stored rows": len(df_raw),
    "Distinct ED encounters": df_raw["A&E Case Number"].nunique(),
    "Distinct patients": df_raw["Patient ID"].nunique(),
}).to_frame("Count"))

# Display the first five rows of the raw data with selected columns
display(df_raw[[
    "A&E Case Number", "Patient ID", "A&E Admit Date Time",
    "A&E Triage Class (Computed)",
]].head(5))

,Count
Stored rows,3002
Distinct ED encounters,3000
Distinct patients,1070


,A&E Case Number,Patient ID,A&E Admit Date Time,A&E Triage Class (Computed)
0,E240000001,PT000085,2024-01-01 07:46:41,P3
1,E240000002,PT000631,2024-01-01 08:00:00,P3
2,E240000003,PT001061,2024-01-01 08:44:52,P3
3,E240000004,PT000788,2024-01-01 13:40:25,P3
4,E240000005,PT000195,2024-01-01 14:36:01,P2


Showing the unique value for A&E Triage Class

In [6]:
# Display the unique values in the "A&E Triage Class (Computed)" column
display(df_raw["A&E Triage Class (Computed)"].dropna().unique())

<StringArray>
['P3', 'P2', 'P1']
Length: 3, dtype: string

## Section 2 — Question 1: Is the ED data clean enough?

Cleaning starts with investigation. A blank can be expected when a field does not apply; a repeated patient can represent a genuine return visit.

### 2.1 Find missing values · Your turn

**Question:** Which fields contain blanks?

**Hint:** `isna()` marks missing values. Add the markers with `sum()`.

In [ ]:
# Check for missing values in the raw data
missing_counts = ____
display(missing_counts[missing_counts > 0].to_frame("Missing rows"))

Which blanks are expected because a field does not apply? Why should we avoid deleting every row with a blank?

### 2.2 Inspect and remove exact duplicate rows

The duplicate check compares all 30 source columns. First inspect every copy, then remove only the extra identical rows from a working copy.

**Hint:** use `df_raw.drop_duplicates().copy()`.

In [ ]:
# Check for exact duplicate rows in the raw data
duplicate_rows = df_raw.duplicated(keep=False)
print("Extra exact copies:", df_raw.duplicated().sum())
display(df_raw.loc[duplicate_rows, [
    "A&E Case Number", "Patient ID", "A&E Admit Date Time",
]])

Extra exact copies: 2


,A&E Case Number,Patient ID,A&E Admit Date Time
296,E240000297,PT000995,2024-03-13 16:23:45
297,E240000297,PT000995,2024-03-13 16:23:45
1974,E250000474,PT000484,2025-04-22 21:01:00
1975,E250000474,PT000484,2025-04-22 21:01:00


In [ ]:
# Remove exact duplicate rows from the raw data
df_clean = ____
print("Rows after exact deduplication:", len(df_clean))
print("Patients retained:", df_clean["Patient ID"].nunique())

Why should repeated patient IDs remain after exact deduplication?

## Section 3 — Question 2: What useful variables can we create?

Data manipulation creates analysis-ready variables without changing the source workbook. We keep `df_raw` unchanged.

### Prepare a smaller working table

In [ ]:
# Rename columns for easier access
column_names = {
    "A&E Case Number": "case_id",
    "Patient ID": "patient_id",
    "Patient Age": "age_years",
    "A&E Admit Date Time": "arrival",
    "A&E Discharge Date Time": "departure",
    "A&E Triage Class (Computed)": "triage",
}

# Create a new DataFrame with renamed columns
df = df_clean[list(column_names)].rename(columns=column_names).copy()
display(df.head(3))

### 3.1 Convert timestamp text

**Question:** Can Python read the arrival and departure values as datetimes?

**Hint:** use `pd.to_datetime(..., errors="coerce")`. Invalid text becomes `NaT`; the audit distinguishes new parsing failures from existing blanks.

In [ ]:
# Parse the arrival and departure columns to datetime
for column in ["arrival", "departure"]:
    source_present = df[column].notna()
    parsed = ____
    print(column, "new parse failures:", (source_present & parsed.isna()).sum())
    df[column] = parsed

### 3.2 Create arrival hour and ED length of stay

**Question:** At what hour did each encounter start, and how many hours did the patient remain in ED?

**Hints:**

- `.dt.hour` extracts the hour from a datetime.
- Subtract arrival from departure, then use `.dt.total_seconds() / 3600`.

In [ ]:
# Create new columns for arrival hour and ED length of stay in hours
df["arrival_hour"] = ____
df["ed_los_hours"] = (
    ____
)
display(df[["case_id", "arrival_hour", "ed_los_hours"]].head(3))

### Check the derived duration

Keep all encounters for attendance counts. Exclude missing and negative durations only from LOS summaries; do not turn them into zero. Ordered long stays remain included.

In [ ]:
# Check for negative or missing length of stay values
negative_los = df["ed_los_hours"] < 0
df["valid_los_hours"] = df["ed_los_hours"].where(df["ed_los_hours"] >= 0)

print("Missing LOS:", df["ed_los_hours"].isna().sum())
print("Negative LOS:", negative_los.sum())
print("All deduplicated encounters:", len(df))
print("Encounters with valid LOS:", df["valid_los_hours"].count())

display(df.loc[negative_los, ["case_id", "arrival", "departure", "ed_los_hours"]])

## Section 4 — Question 3: What patterns can we describe?

Descriptive analysis summarises what appears in the extract. It does not establish why a pattern exists.

We will use:

- a **summary table** for centre and spread;
- a **bar chart** for arrivals by hour;
- a **scatter plot and correlation** for age and ED length of stay.

Always state units and the number of eligible encounters.

### 4.1 Describe ED length of stay

**Question:** What are the mean, median, quartiles, and range of valid ED stays?

**Hint:** call `describe()` on `df["valid_los_hours"]`. In the output, `50%` is the median.

In [ ]:
# Display summary statistics for ED length of stay
los_summary = ____
display(los_summary.round(2).to_frame("ED LOS (hours)"))

Why is the mean larger than the median? Which value better describes a typical stay when the distribution has a long right tail?

### 4.2 Visualise arrivals by hour

**Question:** Which hour has the most ED arrivals?

**Hint:** use `value_counts()` on `df["arrival_hour"]`. The supplied chart code displays all 24 hours.

In [ ]:
# Create a table of encounter counts by arrival hour
hour_counts = ____
hour_counts = hour_counts.reindex(range(24), fill_value=0).sort_index()
hour_table = hour_counts.rename_axis("Arrival hour").reset_index(name="Encounters")

display(hour_table.loc[hour_table["Encounters"].idxmax()].to_frame("Busiest hour"))

In [ ]:
# Create a bar chart of ED encounters by arrival hour
alt.Chart(hour_table).mark_bar(color="#007F86").encode(
    x=alt.X(
        "Arrival hour:O", sort=list(range(24)),
        title="Arrival hour (local time)",
    ),
    y=alt.Y("Encounters:Q", title="ED encounters across 2024–2025"),
    tooltip=["Arrival hour:O", "Encounters:Q"],
).properties(
    width=560, height=240, title="ED arrivals by hour"
)

Are these counts over two years or an average per day?

### 4.3 Visualise age and ED length of stay

**Question:** Do older patients tend to have longer ED stays in this synthetic extract?

The scatter plot shows one point per encounter with both age and valid LOS. Pearson correlation (`r`) summarises a **linear** relationship:

- near `+1`: values tend to increase together;
- near `−1`: one tends to decrease as the other increases;
- near `0`: little linear association.

Correlation does not prove causation. A value near zero does not rule out a curved relationship.

**Hint:** call `.corr()` on the two-column `correlation_data` table.

In [ ]:
# Check for age parsing failures
age_source_present = df["age_years"].notna()
age_numeric = pd.to_numeric(df["age_years"], errors="coerce")
print("Age parse failures:", (age_source_present & age_numeric.isna()).sum())

correlation_data = pd.DataFrame({
    "Age (years)": age_numeric.where(age_numeric >= 0),
    "ED LOS (hours)": df["valid_los_hours"],
}).dropna()

print("Paired encounters:", len(correlation_data))

In [ ]:
# Create a scatter plot of age vs. ED length of stay
alt.Chart(correlation_data).mark_circle(
    size=25, opacity=0.25, color="#007F86"
).encode(
    x=alt.X("Age (years):Q", title="Age at ED encounter (years)"),
    y=alt.Y("ED LOS (hours):Q", title="ED LOS (hours)"),
    tooltip=[
        alt.Tooltip("Age (years):Q", title="Age"),
        alt.Tooltip("ED LOS (hours):Q", title="ED LOS", format=".2f"),
    ],
).properties(
    width=560, height=260, title="Age and ED length of stay"
)

In [ ]:
# Compute the correlation matrix for age and ED length of stay
correlation_matrix = ____
display(correlation_matrix.round(3))

Is the correlation positive, negative, or near zero? What does the scatter plot show that one number cannot?

## Section 5 — Further statistics and visualisation · optional

Complete Sections 1–4 first. Every code cell in this section is ready to run; there are no fill-in blanks.

These examples introduce percentiles, group distributions, bootstrap confidence intervals, and run charts. They remain descriptive learning tools. They do not prove causation or establish hospital performance.

### 5.1 Explore the full LOS distribution

A histogram shows the shape of a numeric distribution. The median describes the middle encounter; the 95th percentile (P95) describes the upper tail. P95 is **not** a target and does not mean 95% of patients should be discharged by that time.

In [ ]:
import numpy as np

valid_los = df["valid_los_hours"].dropna()
los_reference = pd.DataFrame({
    "Statistic": ["Median (P50)", "95th percentile (P95)"],
    "Hours": [valid_los.median(), valid_los.quantile(0.95)],
})
display(los_reference.round(2))

los_histogram = alt.Chart(
    valid_los.rename("ED LOS (hours)").to_frame()
).mark_bar(color="#007F86").encode(
    x=alt.X(
        "ED LOS (hours):Q",
        bin=alt.Bin(step=1),
        title="ED LOS (hours; 1-hour bins)",
    ),
    y=alt.Y("count():Q", title="Encounters with valid LOS"),
    tooltip=[alt.Tooltip("count():Q", title="Encounters")],
)

reference_lines = alt.Chart(los_reference).mark_rule(
    strokeWidth=2, strokeDash=[6, 3]
).encode(
    x=alt.X("Hours:Q"),
    color=alt.Color("Statistic:N", title="Reference"),
    tooltip=[
        alt.Tooltip("Statistic:N"),
        alt.Tooltip("Hours:Q", format=".2f"),
    ],
)

(los_histogram + reference_lines).properties(
    width=600,
    height=280,
    title="ED length-of-stay distribution with median and P95",
)

**Think about it:** Why are both the median and P95 useful when the distribution has a long right tail? What information would be lost if only the mean were reported?

### 5.2 Compare LOS distributions by triage category

A box plot shows the median, the middle 50% of values, and observations beyond the usual whiskers. Points beyond the whiskers are unusual; they are not automatically errors.

The accompanying table provides group sizes and exact quartiles so the chart is not interpreted without denominators.

In [ ]:
triage_los = df[["triage", "valid_los_hours"]].dropna(
    subset=["valid_los_hours"]
).copy()
triage_los["triage"] = triage_los["triage"].fillna("Missing")

triage_los_summary = triage_los.groupby("triage").agg(
    valid_los_n=("valid_los_hours", "count"),
    median_hours=("valid_los_hours", "median"),
    q1_hours=("valid_los_hours", lambda values: values.quantile(0.25)),
    q3_hours=("valid_los_hours", lambda values: values.quantile(0.75)),
)
triage_los_summary["iqr_hours"] = (
    triage_los_summary["q3_hours"] - triage_los_summary["q1_hours"]
)
display(triage_los_summary.round(2))

alt.Chart(triage_los).mark_boxplot(
    extent=1.5, color="#007F86"
).encode(
    x=alt.X(
        "triage:N",
        sort=["P1", "P2", "P3", "Missing"],
        title="Triage category",
    ),
    y=alt.Y("valid_los_hours:Q", title="ED LOS (hours)"),
).properties(
    width=600,
    height=300,
    title="ED length of stay by triage category",
)

**Think about it:** Which category has the highest median? How do unequal group sizes and the very small `Missing` group limit comparison? This is an unadjusted descriptive comparison, not evidence that triage category caused the difference.

### 5.3 Estimate uncertainty with a bootstrap confidence interval

Bootstrapping repeatedly resamples the observed valid LOS values with replacement. The distribution of resampled medians approximates how the sample median could vary across repeated samples.

The interval below is a teaching example. It assumes encounters are independent, although some patients have repeat encounters. It is not a range containing 95% of patient stays.

In [ ]:
rng = np.random.default_rng(42)
bootstrap_medians = np.fromiter(
    (
        np.median(rng.choice(valid_los.to_numpy(), size=len(valid_los), replace=True))
        for _ in range(2000)
    ),
    dtype=float,
)

ci_lower, ci_upper = np.quantile(bootstrap_medians, [0.025, 0.975])
bootstrap_result = pd.DataFrame({
    "Estimate": ["Observed median", "95% CI lower", "95% CI upper"],
    "Hours": [valid_los.median(), ci_lower, ci_upper],
})
display(bootstrap_result.round(2))

bootstrap_frame = pd.DataFrame({"Bootstrap median (hours)": bootstrap_medians})
ci_lines = pd.DataFrame({
    "Boundary": ["Lower 95% CI", "Upper 95% CI"],
    "Hours": [ci_lower, ci_upper],
})

bootstrap_histogram = alt.Chart(bootstrap_frame).mark_bar(
    color="#7A5195"
).encode(
    x=alt.X(
        "Bootstrap median (hours):Q",
        bin=alt.Bin(maxbins=35),
        title="Bootstrap median ED LOS (hours)",
    ),
    y=alt.Y("count():Q", title="Bootstrap samples"),
)

bootstrap_ci = alt.Chart(ci_lines).mark_rule(
    color="#EF8354", strokeWidth=2
).encode(
    x=alt.X("Hours:Q"),
    tooltip=["Boundary:N", alt.Tooltip("Hours:Q", format=".2f")],
)

(bootstrap_histogram + bootstrap_ci).properties(
    width=600,
    height=280,
    title="Bootstrap distribution of the median ED LOS",
)

**Think about it:** A narrow confidence interval describes precision of the estimated median under the resampling assumptions. It does not mean most individual patients have stays inside that interval.

### 5.4 Screen daily volume with a run chart

A run chart preserves time order. The orange points use a descriptive IQR fence to highlight dates for investigation. The fence is not a statistical control limit, a significance test, or proof of an operational problem.

After running the chart, scroll or pinch over it to zoom the date axis, drag left or right to pan, and double-click to reset the view.

In [ ]:
all_dates = pd.date_range(
    df["arrival"].min().normalize(),
    df["arrival"].max().normalize(),
    freq="D",
)
daily_volume = (
    df.assign(arrival_date=df["arrival"].dt.normalize())
      .groupby("arrival_date")
      .size()
      .reindex(all_dates, fill_value=0)
      .rename("encounters")
      .rename_axis("arrival_date")
      .reset_index()
)

q1_daily, median_daily, q3_daily = daily_volume["encounters"].quantile(
    [0.25, 0.50, 0.75]
)
iqr_daily = q3_daily - q1_daily
lower_fence = max(0, q1_daily - 1.5 * iqr_daily)
upper_fence = q3_daily + 1.5 * iqr_daily
daily_volume["screen"] = np.where(
    (daily_volume["encounters"] < lower_fence)
    | (daily_volume["encounters"] > upper_fence),
    "Outside IQR fence",
    "Within fence",
)

daily_line = alt.Chart(daily_volume).mark_line(color="#007F86").encode(
    x=alt.X("arrival_date:T", title="Arrival date"),
    y=alt.Y("encounters:Q", title="ED encounters"),
)
daily_points = alt.Chart(daily_volume).mark_point(
    filled=True, size=45
).encode(
    x="arrival_date:T",
    y="encounters:Q",
    color=alt.Color(
        "screen:N",
        scale=alt.Scale(
            domain=["Within fence", "Outside IQR fence"],
            range=["#007F86", "#EF8354"],
        ),
        title="Descriptive screen",
    ),
    tooltip=[
        alt.Tooltip("arrival_date:T", title="Date"),
        alt.Tooltip("encounters:Q", title="Encounters"),
        "screen:N",
    ],
)
median_line = alt.Chart(
    pd.DataFrame({"Median daily encounters": [median_daily]})
).mark_rule(color="#7A5195", strokeDash=[6, 3]).encode(
    y="Median daily encounters:Q"
)

(daily_line + daily_points + median_line).properties(
    width=700,
    height=300,
    title="Daily ED encounters with a median reference line",
).interactive(bind_y=False)

**Think about it:** An unusual date tells us where to investigate. It does not explain the cause. Check extraction completeness, duplicates, holidays, operational changes, and random variation before drawing a conclusion.